In [1]:
from pygom.model.pygom_model import Model
from pygom.model.transition import Transition
from pygom.model.ode_variable import State

In this notebook, we test that some of the epi_analysis module features work

For now, they have been incorporated into the pygom Model through the maths_methods_store.

In future, we might not think that epi modules have primacy and we should instead import this functionality.

# 1) SIS model

In [2]:
states = [
    State(ID='S', tags=['living']),
    State(ID='I', tags=['living', 'infected', 'infectious'])
]

params = ['beta', 'gamma']
derived_params = [('N', 'S+I')]

inf = Transition(origin='S', destination='I', rate='beta*S*I/N')
rec = Transition(origin='I', destination='S', rate='gamma*I')

event_list = [inf, rec]

mod = Model(state=states, param=params, derived_param=derived_params, event=event_list)

In [3]:
mod.equilibrium.get_equation()

[{I: 0, S: N}, {I: N*(beta - gamma)/beta, S: N*gamma/beta}]

In [4]:
mod.disease_free_equilibrium.get_equation()

[{I: 0, S: N}]

In [5]:
mod.R0.get_equation()

beta/gamma

# 2) SIR model with births and deaths

In [6]:
states = [
    State(ID='S', tags=['living']),
    State(ID='E', tags=['living', 'infected']),
    State(ID='I', tags=['living', 'infected', 'infectious']),
    State(ID='R', tags=['living'])
]

params = ['beta', 'alpha', 'gamma', 'mu']
derived_params = [('N', 'S+E+I+R')]

inf = Transition(origin='S', destination='E', rate='beta*S*I/N')
prog = Transition(origin='E', destination='I', rate='alpha*E')
rec = Transition(origin='I', destination='R', rate='gamma*I')

birth = Transition(destination='S', rate='mu*N')

living_states = ['S', 'E', 'I', 'R']
deaths = [Transition(origin=state, rate=f'mu*{state}') for state in living_states]

event_list = [inf, prog, rec, birth] + deaths

mod = Model(state=states, param=params, derived_param=derived_params, event=event_list)

In [7]:
mod.equilibrium.get_equation()

[{E: 0, I: 0, R: 0, S: N},
 {E: N*mu*(alpha*beta - alpha*gamma - alpha*mu - gamma*mu - mu**2)/(alpha*beta*(alpha + mu)),
  I: N*mu*(alpha*beta - alpha*gamma - alpha*mu - gamma*mu - mu**2)/(beta*(alpha + mu)*(gamma + mu)),
  R: N*gamma*(alpha*beta - alpha*gamma - alpha*mu - gamma*mu - mu**2)/(beta*(alpha + mu)*(gamma + mu)),
  S: N*(alpha + mu)*(gamma + mu)/(alpha*beta)}]

In [8]:
mod.disease_free_equilibrium.get_equation()

[{E: 0, I: 0, R: 0, S: N}]

In [9]:
mod.R0.get_equation()

alpha*beta/((alpha + mu)*(gamma + mu))

# 3) SIR model with births and deaths and asymptomatics

In [10]:
states = [
    State(ID='S', tags=['living']),
    State(ID='E', tags=['living', 'infected']),
    State(ID='I', tags=['living', 'infected', 'infectious']),
    State(ID='A', tags=['living', 'infected', 'infectious']),
    State(ID='R', tags=['living'])
]

params = ['beta', 'alpha', 'gamma', 'mu', 'p', 'r']
derived_params = [('N', 'S+E+I+A+R')]

inf_I = Transition(origin='S', destination='E', rate='beta*S*I/N')
inf_A = Transition(origin='S', destination='E', rate='beta*S*A*r/N')
prog_I = Transition(origin='E', destination='I', rate='(1-p)*alpha*E')
prog_A = Transition(origin='E', destination='A', rate='p*alpha*E')
rec_I = Transition(origin='I', destination='R', rate='gamma*I')
rec_A = Transition(origin='A', destination='R', rate='gamma*A')

birth = Transition(destination='S', rate='mu*N')

living_states = ['S', 'E', 'I', 'A', 'R']
deaths = [Transition(origin=state, rate=f'mu*{state}') for state in living_states]

event_list = [inf_I, inf_A, prog_I, prog_A, rec_I, rec_A, birth] + deaths

mod = Model(state=states, param=params, derived_param=derived_params, event=event_list)

In [11]:
mod.equilibrium.get_equation()

[{A: 0, E: 0, I: 0, R: 0, S: N},
 {A: N*mu*p*(alpha*beta*p*r - alpha*beta*p + alpha*beta - alpha*gamma - alpha*mu - gamma*mu - mu**2)/(beta*(alpha + mu)*(gamma + mu)*(p*r - p + 1)),
  E: N*mu*(alpha*beta*p*r - alpha*beta*p + alpha*beta - alpha*gamma - alpha*mu - gamma*mu - mu**2)/(alpha*beta*(alpha + mu)*(p*r - p + 1)),
  I: -N*mu*(p - 1)*(alpha*beta*p*r - alpha*beta*p + alpha*beta - alpha*gamma - alpha*mu - gamma*mu - mu**2)/(beta*(alpha + mu)*(gamma + mu)*(p*r - p + 1)),
  R: N*gamma*(alpha*beta*p*r - alpha*beta*p + alpha*beta - alpha*gamma - alpha*mu - gamma*mu - mu**2)/(beta*(alpha + mu)*(gamma + mu)*(p*r - p + 1)),
  S: N*(alpha + mu)*(gamma + mu)/(alpha*beta*(p*r - p + 1))}]

In [12]:
mod.disease_free_equilibrium.get_equation()

[{A: 0, E: 0, I: 0, R: 0, S: N}]

In [13]:
mod.R0.get_equation()

alpha*beta*(p*r - p + 1)/((alpha + mu)*(gamma + mu))

# 4) Model with perpetual population growth/decay

This currently won't work, however, we should add a way for the user to say "allow mu_b to be a free parameter such that an equilbrium exists"

In [14]:
states = [
    State(ID='S', tags=['alive']),
    State(ID='E', tags=['alive', 'infected']),
    State(ID='I', tags=['alive', 'infected', 'infectious']),
    State(ID='R', tags=['alive']),
    State(ID='D', tags=['dead', 'cumulative'])
]

params = ['beta', 'alpha', 'gamma', 'mu_b', 'mu_d', 'p']
derived_params = [('N', 'S+E+I+R')]

inf = Transition(origin='S', destination='E', rate='beta*S*I/N')
prog = Transition(origin='E', destination='I', rate='alpha*E')
rec = Transition(origin='I', destination='R', rate='gamma*I*(1-p)')
fatal = Transition(origin='I', destination='D', rate='gamma*I*p')

birth = Transition(destination='S', rate='mu_b*N')

living_states = ['S', 'E', 'I', 'R']
deaths = [Transition(origin=state, rate=f'mu_d*{state}') for state in living_states]

event_list = [inf, prog, rec, fatal, birth] + deaths

mod = Model(state=states, param=params, derived_param=derived_params, event=event_list)

In [15]:
mod.equilibrium.get_equation()

[]

In [16]:
mod.disease_free_equilibrium.get_equation()

[]

In [17]:
mod.R0.get_equation()

[]